# Ручна оцінка перекладу: LAPA vs DeepL

Порядок роботи:
1. **Setup + Data** — завантажити обидва переклади.
2. **Автоматичні перевірки** — швидко знайти очевидні проблеми (пропуски, числа, неперекладені слова).
3. **Перегляд** — подивитися переклади поруч.
4. **Таблиця для розмітки** — створити сліпу таблицю `manual_eval.xlsx` і заповнити її вручну.
5. **Аналіз** — порахувати результати після розмітки.

### Setup

In [1]:
pip install -q pandas openpyxl scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import ast
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", None)

# the notebook runs from its own folder, so walk up to the project root (the folder with .git)
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
TRANSLATED_DIR = PROJECT_ROOT / "data" / "translated"

LAPA_FILE = TRANSLATED_DIR / "PL_train_first_100_uk.csv"
DEEPL_FILE = TRANSLATED_DIR / "PL_train_first_100_uk_deepl.csv"

EVAL_FILE = TRANSLATED_DIR / "manual_eval.xlsx"
KEY_FILE = TRANSLATED_DIR / "manual_eval_key.csv"  # which variant is which system; don't open while annotating
EVAL_FILE_2 = TRANSLATED_DIR / "manual_eval_annotator2.xlsx"  # optional: copy of the sheet filled by a second person

SAMPLE_SIZE = 30
SEED = 42

print("Project root:", PROJECT_ROOT)

Project root: d:\Study\5th_semester\nlp\project\2026_NLP_Course_Project_Detecting_Persuasion_UKRParlTexts


### Data

In [3]:
lapa = pd.read_csv(LAPA_FILE)
deepl = pd.read_csv(DEEPL_FILE)

# both files must contain the same source rows in the same order
assert len(lapa) == len(deepl) and (lapa["content"] == deepl["content"]).all()

df = lapa.drop(columns="content_uk").copy()
df["lapa"] = lapa["content_uk"].fillna("").astype(str)
df["deepl"] = deepl["content_uk"].fillna("").astype(str)
df["techniques"] = df["techniques"].apply(ast.literal_eval)

print(df.shape)
df[["content", "techniques", "lapa", "deepl"]].head()

(100, 14)


,content,techniques,lapa,deepl
0,Ciągle pokutuje mit o Polsce jako kraju tolerancyjnym.,"[Appeal_to_Hypocrisy, Doubt]",Постійно існує міф про Польщу як про толерантну країну.,Досі існує міф про Польщу як про толерантну країну.
1,Rzeczywiście to jest hańba.,"[Exaggeration-Minimisation, Loaded_Language]","Дійсно, це ганьба.",Це справді ганьба.
2,"Posłowie Joanna Borowiak i Anna Gembicka z Klubu Parlamentarnego Prawo i Sprawiedliwość zadadzą pytania w sprawie decyzji generalnego dyrektora ochrony środowiska z grudnia 2024 r. torpedującej budowę stopnia wodnego w Siarzewie, inwestycji zapewnionej przez rząd PiS i gwarantującej bezpieczeństwo przeciwpowodziowe Kujaw.",[Loaded_Language],"Народні депутати Йоанна Боров'як та Анна Гембіцька з парламентського клубу «Право і Справедливість» зададуть питання щодо рішення генерального директора з охорони навколишнього середовища від грудня 2024 року, яке заблокувало будівництво шлюзу в Сіаржеві, інвестиції, забезпеченої урядом PiS та гарантуючої протипаводковий захист Куявії.","Депутати Йоанна Боровіак та Анна Гембіцька з парламентського клубу «Право і справедливість» зададуть запитання щодо рішення генерального директора з питань охорони навколишнього середовища від грудня 2024 року, яке зірвало будівництво гідротехнічної споруди в Сіарзеві — інвестиційного проєкту, забезпеченого урядом партії «Право і Справедливість» та такого, що гарантує захист Куяв від повеней."
3,"Zachęcam do tego, żeby na posiedzeniach tej komisji było mniej hipokryzji ze strony pana posła, a więcej współpracy.",[Questioning_the_Reputation],"Закликаю до того, щоб на засіданнях цієї комісії було менше лицемірства з боку пана депутата, а більше співпраці.","Я закликаю, щоб на засіданнях цієї комісії з боку пана депутата було менше лицемірства, а більше співпраці."
4,"Co prawda tam klimatyzacji nie ma, ale przynajmniej żołnierze w środku nie giną.",[],"Щоправда, там немає кондиціонування, але принаймні солдати всередині не гинуть.","Щоправда, там немає кондиціонера, але принаймні солдати всередині не гинуть."


### Автоматичні перевірки

Не замінюють ручну оцінку, але показують, куди подивитися в першу чергу:
- **len_ratio** — довжина перекладу / довжина оригіналу. Дуже мале значення (< 0.75) часто означає пропуск.
- **numbers_ok** — чи збігаються всі числа з оригіналом.
- **polish_left** — чи лишились польські літери (неперекладений текст).
- **empty** — порожній переклад.

In [4]:
SYSTEMS = ["lapa", "deepl"]
NUM_RE = re.compile(r"\d+(?:[.,]\d+)?")
POLISH_RE = re.compile(r"[ąćęłńóśźżĄĆĘŁŃÓŚŹŻ]")


def numbers(text):
    return sorted(n.replace(",", ".") for n in NUM_RE.findall(text))


for s in SYSTEMS:
    df[f"{s}_len_ratio"] = df[s].str.len() / df["content"].str.len()
    df[f"{s}_numbers_ok"] = [numbers(src) == numbers(mt) for src, mt in zip(df["content"], df[s])]
    df[f"{s}_polish_left"] = df[s].str.contains(POLISH_RE)
    df[f"{s}_empty"] = df[s].str.strip() == ""

summary = pd.DataFrame({
    s: {
        "mean len_ratio": df[f"{s}_len_ratio"].mean().round(2),
        "len_ratio < 0.75": (df[f"{s}_len_ratio"] < 0.75).sum(),
        "numbers mismatch": (~df[f"{s}_numbers_ok"]).sum(),
        "polish letters left": df[f"{s}_polish_left"].sum(),
        "empty": df[f"{s}_empty"].sum(),
    }
    for s in SYSTEMS
})
summary

,lapa,deepl
mean len_ratio,0.94,1.0
len_ratio < 0.75,5.00,2.0
numbers mismatch,0.00,0.0
polish letters left,0.00,0.0
empty,0.00,0.0


In [5]:
# rows where at least one system looks suspicious
suspicious = (
    (df["lapa_len_ratio"] < 0.75) | (df["deepl_len_ratio"] < 0.75)
    | ~df["lapa_numbers_ok"] | ~df["deepl_numbers_ok"]
    | df["lapa_polish_left"] | df["deepl_polish_left"]
    | df["lapa_empty"] | df["deepl_empty"]
)
print("Suspicious rows:", suspicious.sum())
df.loc[suspicious, ["content", "lapa", "deepl", "lapa_len_ratio", "deepl_len_ratio"]]

Suspicious rows: 6


,content,lapa,deepl,lapa_len_ratio,deepl_len_ratio
1,Rzeczywiście to jest hańba.,"Дійсно, це ганьба.",Це справді ганьба.,0.666667,0.666667
10,Oczywiście to jest wasza stara śpiewka.,"Звичайно, це ваша стара пісня.","Звісно, це ваша стара пісня.",0.769231,0.717949
30,Zaledwie 4% Polaków ma szansę na znalezienie bezpiecznego schronienia.,Лише 4% поляків мають шанс знайти безпечне укриття.,Лише 4% поляків мають шанс знайти безпечний притулок.,0.728571,0.757143
33,Wskazywałem na samym początku dlaczego.,"Я вказував на початку, чому.","Я ще на самому початку пояснював, чому.",0.717949,1.000000
34,Poseł Janusz Kowalski: Szanowny Panie Marszałku!,Шановний пане Маршалку!,Депутат Януш Ковальський: Шановний пане Маршалку!,0.479167,1.020833
67,"Jest to kobieta po czterdziestce, z jaskrą.","Це жінка за сорок, з веснянками.","Це жінка за сорок, яка страждає на глаукому.",0.744186,1.023256


### Перегляд перекладів поруч

In [6]:
def show(i):
    row = df.loc[i]
    print(f"[{i}] techniques: {row['techniques']}")
    print("PL:   ", row["content"])
    print("LAPA: ", row["lapa"])
    print("DeepL:", row["deepl"])
    print("-" * 100)


for i in df.sample(5, random_state=SEED).index:
    show(i)

[83] techniques: ['False_Dilemma-No_Choice']
PL:    Debatujemy dzisiaj o bardzo ważnej i – ośmielę się powiedzieć – kluczowej dla nas sprawie, sprawie życia i śmierci.
LAPA:  Сьогодні ми обговорюємо дуже важливу і – дозволю собі сказати – ключову для нас справу, справу життя і смерті.
DeepL: Сьогодні ми обговорюємо дуже важливе і — смію сказати — ключове для нас питання, питання життя і смерті.
----------------------------------------------------------------------------------------------------
[53] techniques: ['Loaded_Language', 'Flag_Waving', 'Questioning_the_Reputation', 'Name_Calling-Labeling']
PL:    Dzisiaj każda polityczna wypowiedź musi zaczynać się od przypomnienia wszystkim raz jeszcze – chodzi też o to, żebyśmy mieli odwagę wypowiedzieć to także na forum międzynarodowym – że to Putin, że to Rosja napadła Ukrainę, że to jest część rosyjskiej polityki imperialnej, od kilkuset lat wymierzonej w nasze państwo i inne państwa i narody Europy Środkowej.
LAPA:  Сьогодні кожна політи

### Таблиця для розмітки

Створює `data/translated/manual_eval.xlsx`. У ній переклади перемішані й підписані тільки як **A** / **B**, щоб оцінювати наосліп. Яка система де, записано в `manual_eval_key.csv`. Не відкривайте цей файл до кінця розмітки.

Підозрілі рядки з автоматичних перевірок завжди потрапляють у вибірку, решту добирає випадкова вибірка.

Якщо таблиця вже існує, клітинка її **не перезаписує**, щоб не втратити розмітку. Щоб створити нову, видаліть старий файл вручну.

**Як заповнювати** (для кожного рядка):

| Колонка | Значення | Питання |
|---|---|---|
| `adequacy_1_5` | 1–5 | Чи переданий весь зміст: без пропусків, доданого, спотворень? |
| `fluency_1_5` | 1–5 | Чи звучить як природна українська парламентська мова? |
| `technique_preserved` | `yes` / `partly` / `no` | Чи збережена техніка з колонки `techniques` (емоційна лексика, іронія, перебільшення…)? |
| `error_type` | `none` / `mistranslation` / `omission` / `addition` / `names_terms` / `grammar` / `style` | Головний тип помилки |
| `comment` | текст | Короткий приклад проблеми |

Шкала adequacy: **5** — зміст передано повністю, **4** — дрібні неточності, **3** — частина змісту втрачена, **2** — серйозні спотворення, **1** — переклад не має сенсу.

In [7]:
if EVAL_FILE.exists():
    print(f"{EVAL_FILE.name} already exists - not overwriting. Delete it manually to create a new one.")
else:
    rng = np.random.default_rng(SEED)

    forced = df.index[suspicious].tolist()[:SAMPLE_SIZE]
    rest = df.index.difference(forced)
    sampled = rng.choice(rest, size=SAMPLE_SIZE - len(forced), replace=False).tolist()
    idx = rng.permutation(forced + sampled)

    rows, key = [], []
    for i in idx:
        pair = [("LAPA", df.loc[i, "lapa"]), ("DeepL", df.loc[i, "deepl"])]
        rng.shuffle(pair)
        for variant, (system, text) in zip(["A", "B"], pair):
            rows.append({
                "row_id": i,
                "variant": variant,
                "content_pl": df.loc[i, "content"],
                "techniques": ", ".join(df.loc[i, "techniques"]),
                "content_uk": text,
                "adequacy_1_5": "",
                "fluency_1_5": "",
                "technique_preserved": "",
                "error_type": "",
                "comment": "",
            })
            key.append({"row_id": i, "variant": variant, "system": system})

    pd.DataFrame(rows).to_excel(EVAL_FILE, index=False)
    pd.DataFrame(key).to_csv(KEY_FILE, index=False)
    print(f"Saved {len(rows)} rows ({len(idx)} sentences x 2 systems) to {EVAL_FILE}")

Saved 60 rows (30 sentences x 2 systems) to d:\Study\5th_semester\nlp\project\2026_NLP_Course_Project_Detecting_Persuasion_UKRParlTexts\data\translated\manual_eval.xlsx


### Аналіз (після розмітки)

Запускайте, коли заповните `manual_eval.xlsx`. Незаповнені рядки ігноруються, тож можна дивитися проміжні результати.

In [8]:
def load_eval(path):
    ev = pd.read_excel(path).merge(pd.read_csv(KEY_FILE), on=["row_id", "variant"])
    for col in ["adequacy_1_5", "fluency_1_5"]:
        ev[col] = pd.to_numeric(ev[col], errors="coerce")
    for col in ["technique_preserved", "error_type"]:
        ev[col] = ev[col].astype(str).str.strip().str.lower().replace({"nan": np.nan, "": np.nan})
    return ev


ev = load_eval(EVAL_FILE)
done = ev["adequacy_1_5"].notna()
print(f"Annotated: {done.sum()} / {len(ev)}")

ev.groupby("system")[["adequacy_1_5", "fluency_1_5"]].agg(["mean", "std", "count"]).round(2)

Annotated: 0 / 60


C:\Users\Admin\AppData\Local\Temp\ipykernel_12688\486212796.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ev[col] = ev[col].astype(str).str.strip().str.lower().replace({"nan": np.nan, "": np.nan})
C:\Users\Admin\AppData\Local\Temp\ipykernel_12688\486212796.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ev[col] = ev[col].astype(str).str.strip().str.lower().replace({"nan": np.nan, "": np.nan})


adequacy_1_5           fluency_1_5          
               mean std count        mean std count
system                                             
DeepL           NaN NaN     0         NaN NaN     0
LAPA            NaN NaN     0         NaN NaN     0

In [9]:
# share of sentences where the persuasion technique survived the translation
pd.crosstab(ev["system"], ev["technique_preserved"], normalize="index").round(2)

technique_preserved
system


In [10]:
# most frequent error types
pd.crosstab(ev["error_type"], ev["system"])

system
error_type


In [11]:
# head-to-head: which system got the higher adequacy on the same sentence
wide = ev.pivot_table(index="row_id", columns="system", values="adequacy_1_5").dropna()
print("LAPA better: ", (wide["LAPA"] > wide["DeepL"]).sum())
print("DeepL better:", (wide["DeepL"] > wide["LAPA"]).sum())
print("Tie:         ", (wide["LAPA"] == wide["DeepL"]).sum())

KeyError: 'LAPA'

In [ ]:
# the worst translations - good examples for the report
ev[done].sort_values("adequacy_1_5")[
    ["system", "content_pl", "content_uk", "adequacy_1_5", "error_type", "comment"]
].head(10)

### Узгодженість анотаторів (необов'язково)

Якщо друга людина незалежно заповнила копію таблиці й зберегла її як `manual_eval_annotator2.xlsx`, тут рахується Cohen's kappa. Для шкали 1–5 використовується квадратично зважена kappa. Орієнтир: > 0.6 — добра узгодженість.

In [ ]:
from sklearn.metrics import cohen_kappa_score

if not EVAL_FILE_2.exists():
    print(f"{EVAL_FILE_2.name} not found - skipping.")
else:
    ev2 = load_eval(EVAL_FILE_2)
    both = ev.merge(ev2, on=["row_id", "variant"], suffixes=("_1", "_2"))
    for col, weights in [("adequacy_1_5", "quadratic"), ("fluency_1_5", "quadratic"), ("technique_preserved", None)]:
        pair = both[[f"{col}_1", f"{col}_2"]].dropna()
        kappa = cohen_kappa_score(pair[f"{col}_1"], pair[f"{col}_2"], weights=weights)
        print(f"{col}: kappa = {kappa:.2f} (n = {len(pair)})")